In [76]:
import yfinance as yf

In [77]:
import numpy as np

In [78]:
import math

## Stock Option Inputs

In [81]:
ticker_symbol = "MSFT"

In [82]:
ticker = yf.Ticker(ticker_symbol)

In [83]:
option_type = "call"

In [84]:
strike_price = 400

In [85]:
time_to_expiry = 81 / 365

# Retrieve supporting data


## Fetch 3-month T-bill yield (^IRX)


In [86]:
t_bill = yf.Ticker("^IRX") 

### Get recent historical data for T-bill

In [87]:
hist = t_bill.history(period="5d")  # last 5 days

In [88]:
risk_free_rate = hist['Close'].iloc[-1] / 100  # Convert % to decimal

In [89]:
print(f"Latest 3-month T-bill yield (risk-free rate): {risk_free_rate:.4%}")

Latest 3-month T-bill yield (risk-free rate): 3.5880%


In [90]:
print(risk_free_rate)

0.03588000059127808


In [91]:
time_to_expiry = 81 / 365

In [92]:
current_price = ticker.info['currentPrice']

In [93]:
print(current_price)

397.2


In [94]:
data = yf.download(ticker_symbol, period="60d", interval="1d", progress=False)

In [95]:
close_prices = data["Close"]

## Supporting transformations and calculations 

In [98]:
spot_price = close_prices.iat[-1, 0]

In [99]:
returns = np.log(close_prices.MSFT / close_prices.MSFT.shift(1)).dropna()

In [100]:
volatility = np.std(returns) * np.sqrt(252)

In [101]:
d1 = (np.log(spot_price / strike_price) + (risk_free_rate + 0.5 * volatility ** 2) * time_to_expiry) / (volatility * np.sqrt(time_to_expiry))

In [102]:
d2 = d1 - volatility * np.sqrt(time_to_expiry)

In [103]:
def e_power_neg_x_squared(x):
    # Validate input type
    if not isinstance(x, (int, float)):
        raise TypeError("x must be an integer or float")
    return math.exp(-x**2)

In [104]:
def erf(x):
    # save the sign of x
    sign = 1 if x >= 0 else -1
    x = abs(x)

    # constants
    a1 =  0.254829592
    a2 = -0.284496736
    a3 =  1.421413741
    a4 = -1.453152027
    a5 =  1.061405429
    p  =  0.3275911

    # A&S formula 7.1.26
    t = 1.0/(1.0 + p*x)
    y = 1.0 - (((((a5*t + a4)*t) + a3)*t + a2)*t + a1)*t*math.exp(-x*x)
    return sign*y # erf(-x) = -erf(x)

In [105]:
if option_type == "call":
    option_price = spot_price * np.exp(-0) * 0.5 * (1 + erf(d1 / np.sqrt(2))) - strike_price * np.exp(-risk_free_rate * time_to_expiry) * 0.5 * (1 + erf(d2 / np.sqrt(2)))
    delta = 0.5 * (1 + erf(d1 / np.sqrt(2)))
else:
    option_price = strike_price * np.exp(-risk_free_rate * time_to_expiry) * 0.5 * (1 - erf(d2 / np.sqrt(2))) - spot_price * np.exp(-0) * 0.5 * (1 - erf(d1 / np.sqrt(2)))
    delta = -0.5 * (1 - erf(d1 / np.sqrt(2)))

In [106]:
gamma = np.exp(-d1 ** 2 / 2) / (spot_price * volatility * np.sqrt(2 * np.pi * time_to_expiry))

## Output 1 - MSFT Call

In [107]:
print ([option_price, delta, gamma])

[np.float64(23.643542571134475), np.float64(0.5320886490952832), np.float64(0.006754372748216858)]


In [108]:
option_type = "put"

In [109]:
if option_type == "call":
    option_price = spot_price * np.exp(-0) * 0.5 * (1 + erf(d1 / np.sqrt(2))) - strike_price * np.exp(-risk_free_rate * time_to_expiry) * 0.5 * (1 + erf(d2 / np.sqrt(2)))
    delta = 0.5 * (1 + erf(d1 / np.sqrt(2)))
else:
    option_price = strike_price * np.exp(-risk_free_rate * time_to_expiry) * 0.5 * (1 - erf(d2 / np.sqrt(2))) - spot_price * np.exp(-0) * 0.5 * (1 - erf(d1 / np.sqrt(2)))
    delta = -0.5 * (1 - erf(d1 / np.sqrt(2)))

In [110]:
gamma = np.exp(-d1 ** 2 / 2) / (spot_price * volatility * np.sqrt(2 * np.pi * time_to_expiry))

## Output 2 - MSFT Put

In [111]:
print ([option_price, delta, gamma])

[np.float64(23.266237973328373), np.float64(-0.46791135090471675), np.float64(0.006754372748216858)]
